# Project:

## **Project Parts:**

* data loading
* PCA exploration
* LDA exploration
* classification
* threshold tuning
* PCA + LDA experiments.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg as la
from Utils.utils import load, split_db, compute_pca, apply_pca, compute_lda, apply_lda, vcol, vrow

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True


## 1. Load and split the data

* The project dataset has `6` features and `2` classes.

* We split it into a `training` partition and a `validation` partition using the provided random split.

In [ ]:
D, L = load('../Data/trainData.txt')
(DTR, LTR), (DVAL, LVAL) = split_db(D, L, seed=0)
# D.shape, np.bincount(L), DTR.shape, DVAL.shape


## 2. PCA exploration on the full project data

* PCA is used here only for exploratory analysis.
* The full 6-dimensional PCA is a rotation of the feature space, so it can change the visual appearance of each marginal distribution without changing the underlying geometry.

In [ ]:
def plot_pca_histograms(D, L, m=6, title='PCA histograms'):
    P = compute_pca(D, m)
    DP = apply_pca(D, P)
    fig, axs = plt.subplots(2, 3, figsize=(16, 9))
    axs = axs.ravel()
    for i in range(m):
        ax = axs[i]
        ax.hist(DP[i, L==0], bins=50, density=True, alpha=0.55, label='Class 0')
        ax.hist(DP[i, L==1], bins=50, density=True, alpha=0.55, label='Class 1')
        ax.set_title(f'PCA direction {i+1}')
        ax.legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()
    return P, DP

P_full, D_pca_full = plot_pca_histograms(D, L, m=6, title='PCA histograms on the project data')


## PCA Does Not Necessarily Find the Best Discriminant Direction

PCA looks for the direction of **maximum variance** in the data — it has no knowledge of the class labels. So when we ask "is this direction good for separating the two classes?", the answer depends entirely on how the data is structured, not on any property of PCA itself.

To understand why, recall the LDA criterion. LDA searches for the direction **w** that maximizes the ratio:

$$
\frac{S_B}{S_W}
$$

where $S_B$ is the **between-class variance** (how far apart the class means are along **w**) and $S_W$ is the **within-class variance** (how spread out the samples are around their class mean along **w**). A large ratio means the classes are well separated relative to their spread — exactly what we want for classification.

PCA instead maximizes only the **total variance** $S_T$, which is:

$$
S_T = S_B + S_W
$$

So PCA maximizes $S_B + S_W$, while LDA maximizes $S_B / S_W$. These are completely different objectives, and in general they will find different directions.

***

## When Are PCA and LDA Guaranteed to Find the Same Direction?

There is one special case where PCA and LDA are **equivalent**: when the **within-class covariance matrix $S_W$ is proportional to the identity matrix** — meaning the within-class variance is the **same in every direction**.

Here is why: if $S_W = c \cdot I$ (a constant times the identity), then for any direction **w**:

$$
\frac{S_B}{S_W} = \frac{S_B}{c}
$$

Since $c$ is just a constant, maximizing $S_B / S_W$ is the same as maximizing $S_B$ alone. But we also know:

$$
S_T = S_B + S_W = S_B + c
$$

So maximizing $S_B$ is the same as maximizing $S_T$. And **PCA maximizes $S_T$**. Therefore, in this special case, **PCA and LDA find the same solution**.

For this project dataset, the professor observed that the within-class covariance matrix is approximately diagonal, with similar values along the diagonal — meaning the within-class variance is roughly equal across all features. This is why PCA happened to give a direction that is also discriminative. **It was not by design — it was a consequence of the data geometry.**

***

## PCA Is Just a Rotation — It Does Not Destroy Clusters

When you apply PCA with all 6 directions (full dimensionality), you are performing a **pure rotation** of the data. The distances between points, the shape of the clusters, and the separation between classes do not change at all. They only look different because you are viewing the data from a different angle.

This is why, even if the histograms of the PCA-transformed features look different from the original feature histograms, the clusters have not actually changed. The histogram is a 1D projection — it can only show you one direction at a time, so it may hide structure that is visible in the full space.

***


## PCA Highlights Directions — It Does Not Create Separation

This is the most important conceptual point: **PCA does not increase the separation between classes**. The separation is already inside the data. What PCA (and LDA) do is find and highlight the directions along which that separation already exists.

If after applying LDA or PCA you see that the classes are well separated in the projected histogram, it is not because the method created that separation — it is because that direction was already discriminative in the original feature space, and the method brought it to the forefront.

> **In short:** PCA finds directions of maximum variance. LDA finds directions of maximum class separation. They coincide only when the within-class spread is the same everywhere. On this dataset, that condition approximately holds — which is why PCA appears to work well, but this is not a general guarantee.

### **Prof Notes**

- The professor stressed that from the 1D histograms you can only say what is visible in that projection, not the full structure of the dataset.

- In the scatter plots, you may see more clusters than in the histograms, but even then you still only see the dataset from one “viewing angle.”

- The professor said that applying PCA with all 6 directions does not change the global structure of the dataset — it is just a rotation. So if clusters look different in the PCA plots, that is only a change of viewpoint, not a real change in the data geometry.

- He emphasized that PCA does not search for class separation. It searches for directions with maximum variance, so if a PCA direction looks useful for classification, that is because of how this particular dataset is arranged, not because PCA is inherently a classifier-friendly method.

- He explicitly said PCA may work by chance on this data, because the direction of highest variance happens to also separate the classes. That is not a general rule.

- PCA does not make the classes more separated. The separation is already in the data; PCA only highlights a direction where that separation is visible. If you remove dimensions, you are throwing away information, not creating new separability.

## 3. LDA exploration
LDA explicitly searches for the direction that maximizes class separation relative to within-class spread. Because the direction is defined only up to sign, the code below flips it when needed so class 1 has the larger projected mean.

In [ ]:
def lda_1d_train(DTR, LTR):
    W = compute_lda(DTR, LTR, 1)
    DTR_lda = apply_lda(W, DTR)
    mu0 = DTR_lda[0, LTR==0].mean()
    mu1 = DTR_lda[0, LTR==1].mean()
    if mu1 < mu0:
        W = -W
        DTR_lda = -DTR_lda
        mu0, mu1 = -mu0, -mu1

    # The Threshold is in the middle of the mean of 2 classes    
    thr = (mu0 + mu1) / 2.0
    return W, DTR_lda, thr

In [ ]:

def plot_lda_histograms(D, L, title='LDA histogram'):
    W, D_lda, thr = lda_1d_train(D, L)
    plt.figure(figsize=(10, 5))
    plt.hist(D_lda[0, L==0], bins=30, density=True, alpha=0.55, label='Class 0')
    plt.hist(D_lda[0, L==1], bins=30, density=True, alpha=0.55, label='Class 1')
    plt.axvline(thr, color='k', linestyle='--', label=f'threshold={thr:.3f}')
    print("Thr: ", thr, "Which is the average of the mean of the 2 classes.")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()
    return W, D_lda, thr

W_full, D_lda_full, thr_full = plot_lda_histograms(D, L, title='1D LDA on the full project data')


### **Prof Notes:**

This is the most important theoretical point he made for this part:

- if the within-class variance is the same in every direction — in other words, if the within-class covariance is effectively a constant / scaled identity — then maximizing the LDA ratio becomes equivalent to maximizing the between-class separation. In that special case, PCA and LDA can end up finding a similar solution.

- For this dataset, he said the covariance structure makes this approximation reasonable:

    - The covariance of each class is almost diagonal, and the within-class variance is very similar across features. That is why PCA happened to behave well here.


- **Features 1, 2, 5, and 6 are weak for LDA**

    - He said the means of classes in features 1, 2, 5, and 6 are very close, so those features have little between-class discriminative power. Even if you combine them, the class means will still not move far apart enough to be strongly useful.

- **Features 3 and 4 carry the separation**

    - For features 3 and 4, the class means are farther apart, and the within-class variance is still similar. So LDA should place more weight on those directions. In the special case where within-class variance is the same for all directions, the best LDA solution becomes the direction that connects the class means.

## 4. LDA classification on the validation split

* The classifier is trained on the training split only, then applied to the validation split.
* Predictions are obtained by comparing projected samples to the threshold defined by the projected class means on the training data.

In [ ]:
# The thr here is the middle of the mean of the 2 classes
W, DTR_lda, thr = lda_1d_train(DTR, LTR)

DVAL_lda = apply_lda(W, DVAL)

# Doing Classification on the Validation DataSet
PVAL = np.zeros_like(LVAL)
PVAL[DVAL_lda[0] >= thr] = 1
PVAL[DVAL_lda[0] < thr] = 0

# Error Rate
err = (PVAL != LVAL).sum()
err_rate = err / LVAL.size

print("Number of Validation Samples: ", LVAL.size)
print("Number of Wrong Predictions: ", err)
err_rate_str = str(err_rate * 100) + "%"
print("Error Rate: ", err_rate_str)


## 5. Threshold sweep

The threshold is not fixed by LDA itself, so it is useful to inspect nearby values and see whether validation accuracy improves. The sweep below searches thresholds between the minimum and maximum validation projections.

In [ ]:
# All the Candidate Thr, so we can choose the best one
cand_thr = np.linspace(DVAL_lda.min(), DVAL_lda.max(), 2000)

errs = []
for t in cand_thr:
    pred = np.zeros_like(LVAL)
    pred[DVAL_lda[0] >= t] = 1
    errs.append((pred != LVAL).mean())
best_idx = int(np.argmin(errs))
best_thr = cand_thr[best_idx]
best_err = errs[best_idx]

print("Default Threshold:\t", thr)          # ( Meu(C1) + Meu(C2) ) / 2
print("Best Threshold:\t\t", best_thr)
print("-----------------")
print("DVal_lda Min: ", DVAL_lda.min())
print("DVAL_lda Max: ", DVAL_lda.max())
best_err_str = str(best_err*100) + "%"
print("Min Error: ", best_err_str)


## 6. PCA before LDA

- **Steps:**

    1. PCA is estimated only on the training split
    2. LDA is trained in the reduced space. 

This section compares validation error for several values of the **PCA** dimensionality `m`.

In [ ]:
def pca_lda_eval(DTR, LTR, DVAL, LVAL, m):
    P = compute_pca(DTR, m)
    DTR_p = apply_pca(DTR, P)
    DVAL_p = apply_pca(DVAL, P)
    W = compute_lda(DTR_p, LTR, 1)
    DTR_l = apply_lda(W, DTR_p)
    DVAL_l = apply_lda(W, DVAL_p)
    
    mu0 = DTR_l[0, LTR==0].mean()
    mu1 = DTR_l[0, LTR==1].mean()
    if mu1 < mu0:
        W = -W
        DTR_l = -DTR_l
        DVAL_l = -DVAL_l
        mu0, mu1 = -mu0, -mu1

    # Middle of mean 2 classes
    thr = (mu0 + mu1) / 2.0
    pred = np.zeros_like(LVAL)
    pred[DVAL_l[0] >= thr] = 1
    err = (pred != LVAL).mean()
    return err, thr, P, W

results = []
for m in [2, 3, 4, 5, 6]:
    err, thr_m, _, _ = pca_lda_eval(DTR, LTR, DVAL, LVAL, m)
    results.append((m, err, thr_m))


for en, result in enumerate(results):
    err_str = str(result[1] * 100) + "%"
    print(str(en+1)+"-" ,  "m:", result[0], "| Error: ",  err_str, "| Threshold: ", result[2])
    


### **Prof Notes:**

- **Note 1 — PCA before LDA:**

    PCA before LDA is not necessarily beneficial. With all 6 dimensions it changes nothing (it is just a rotation). With fewer dimensions it may or may not help depending on whether the removed directions contained useful discriminative information or just noise. You must test it experimentally on the validation set.

<br>

- **Note 2 — PCA uncorrelates globally, not inside each class:**

    PCA diagonalizes the total covariance matrix $S_T$ — all samples together. Naive Bayes needs the within-class covariance $S_{W,c}$ of each individual class to be diagonal. These are two completely different matrices, and making one diagonal does not make the other diagonal.

<br>

- **Note 3 — PCA can make class-conditional structure worse:**

    On this dataset, before PCA, the within-class covariance was already approximately diagonal — features were already approximately independent inside each class. After PCA, the rotation introduced within-class correlations that were not there before. So the global structure got cleaner, but the class-level structure that Naive Bayes depends on got worse.

<br>

- **Note 4 — Why the error rate jumped:**

    Naive Bayes always assumes independence inside each class — it never checks, it just assumes. When the data satisfied that assumption approximately (before PCA), it worked well. After PCA introduced within-class correlations, Naive Bayes was still applying the same independence assumption to data where that assumption was now wrong. The result was a jump from ~7% to ~29% error rate — exactly the effect the professor demonstrated.


# Project Questions and Answers

## 1. PCA on the project data

PCA is useful here for exploratory analysis, but it is not designed to maximize class separation. The first principal components are the directions along which the data has the largest variance, so the PCA histograms mainly tell us how the data is spread, not how well the two classes can be separated.

When PCA is applied while keeping all 6 dimensions, the transformation is only a rotation of the original feature space. This means the underlying geometry of the dataset does not change, even if some projected histograms or scatter plots look more structured or visually cleaner.

So, if some PCA directions appear to show less overlap, this does not mean PCA has really created better class separation. It only means that the same data structure is being viewed from a different angle.



## 2. What do the PCA histograms show?

The PCA histograms show how each class is distributed along each principal direction. The first directions usually contain most of the variance, while later directions carry less information and often look noisier or flatter.

A useful observation is that different PCA components may make some clusters or local structures easier to see. However, this is still only an exploratory effect, because PCA does not use the labels and therefore does not explicitly try to separate class 0 from class 1.

So the correct interpretation is that PCA can help us inspect the structure of the data, but it does not guarantee better discrimination between the classes.



## 3. Have the clusters really changed after PCA?

No, the clusters have not fundamentally changed after PCA. If all 6 dimensions are retained, PCA only rotates the coordinate system, so the relative positions of the samples stay the same.

This means the apparent changes in the histograms or scatter plots are visual changes, not real geometric changes. The classes are not inherently becoming more separated, and the clusters are not being newly created by PCA.

Therefore, PCA can change how easy it is to *see* a pattern in a single projection, but it does not change the true global structure of the dataset.



## 4. LDA on the project data

LDA is more appropriate for this classification task because it is supervised. Unlike PCA, it uses the class labels and searches for the direction that maximizes the separation between class means while reducing the spread inside each class.

For this reason, the 1D LDA projection should show less overlap between the two classes than the first PCA direction. In other words, LDA is directly optimized for discrimination, while PCA is optimized only for variance.

This is why LDA is usually a better choice than PCA when the main goal is classification rather than visualization or compression.



## 5. What do you observe in the LDA histogram?

The LDA histogram should show the two classes distributed along a single direction with less overlap than in the original features or in the first PCA component. This indicates that LDA has found a direction where the classes are easier to separate with a threshold.

If the overlap is small, then the LDA direction is a good candidate for classification. If there is still some overlap, then errors will mainly happen for the points close to the decision boundary.

So the main conclusion is that LDA is finding a more discriminative projection than PCA for the binary classification problem.



## 6. Why do we fix the sign of the LDA direction?

The sign of the LDA direction is arbitrary, because multiplying the projection vector by -1 gives the same subspace. However, for classification we need a consistent orientation so that the threshold assigns labels correctly.

A standard way to do this is to flip the LDA direction if necessary so that the projected mean of class 1 is greater than the projected mean of class 0. This makes the classification rule easy to define and interpret.

Without this sign correction, the classifier may assign the classes in the opposite direction even if the projection itself is mathematically correct.



## 7. How is classification performed with LDA?

First, LDA is estimated using only the training split. Then both the training and validation samples are projected onto the learned 1D LDA direction.

Next, the threshold is chosen as the midpoint between the projected mean of class 0 and the projected mean of class 1 on the training set. Validation samples are then classified according to whether they fall to the left or to the right of this threshold.

This gives a simple linear classifier in one dimension, and its performance is measured by counting the number of validation errors.

## 8. What does the validation result mean?

The validation result tells us how well the learned LDA direction generalizes to unseen samples. Since the validation set is not used to estimate the projection, the error rate is a more realistic estimate of performance than training accuracy.

If the error rate is low, then the LDA direction is capturing a real structure in the data rather than memorizing the training set. If the error rate is higher than expected, then the main things to inspect are the class overlap and the threshold choice.

So the validation step is important because it checks whether the classifier actually works on new data and not only on the data used to train it.

## 9. What happens if we change the threshold?

The midpoint threshold is a simple baseline, but it is not always the best possible threshold. By moving the threshold slightly left or right, some borderline samples may be classified more accurately.

This means that a threshold sweep can sometimes improve validation accuracy. If this happens, it suggests that the class distributions are not perfectly balanced around the midpoint, even though the LDA direction itself is still useful.

So the threshold controls the final decision rule, and even with a good projection, classification performance can still depend on where the threshold is placed.


## 10. PCA before LDA

Applying PCA before LDA can be helpful when the original feature space contains noisy or redundant directions. In that case, PCA may simplify the data and allow LDA to work in a lower-dimensional space.

However, PCA is unsupervised, so it can also remove directions that have low variance but still contain useful class information. Because of this, PCA before LDA is not guaranteed to improve classification accuracy.

The correct way to evaluate this is to test several PCA dimensions \(m\) on the validation set and compare the error rates. The best value of \(m\) is the one that gives the lowest validation error, not necessarily the one with the largest explained variance.








## 11. Is PCA beneficial when combined with LDA?

PCA can be beneficial, but only in some cases. If it removes mostly noise and redundancy, it may slightly improve LDA performance. If it removes discriminative information, then it can reduce the quality of the classifier.

So PCA should be treated as a preprocessing option that must be validated experimentally. It is not automatically useful, and the final answer depends on the validation results for different values of \(m\).

The best conclusion is that LDA is the main supervised tool for this task, while PCA is optional and helpful only if it improves validation performance.



## 12. Final conclusion

PCA and LDA serve different purposes. PCA is mainly useful for exploratory analysis and dimensionality reduction based on variance, while LDA is designed to find directions that separate classes.

For this project, LDA is more effective than PCA for the binary classification task because it explicitly uses the labels. PCA is still useful for visualization and as optional preprocessing, but it should not be expected to outperform LDA by itself for class separation.